In [0]:
from pyspark.sql import functions as F

# 1. Load source trips
df_jan = spark.read.parquet("/Volumes/urban_mobility/bronze/source_data/yellow_tripdata_2026-01.parquet")
df_feb = spark.read.parquet("/Volumes/urban_mobility/bronze/source_data/yellow_tripdata_2026-02.parquet")

source_df = df_jan.unionByName(df_feb)

total_count = source_df.count()
TARGET_TRIPS = 1_800_000
sample_fraction = min(1.0, TARGET_TRIPS / total_count)

sampled_df = source_df.sample(fraction=sample_fraction, seed=42)
print("Source trips:", total_count, "| Sampled trips:", sampled_df.count())

# 2. Drivers/Vehicles pairs
vehicles_tbl = spark.table("urban_mobility.bronze.vehicles").select("assigned_driver_id", "vehicle_id")
pairs_list = [(r.assigned_driver_id, r.vehicle_id) for r in vehicles_tbl.collect()]
n_pairs = len(pairs_list)
pairs_df = spark.createDataFrame(
    [(i, d, v) for i, (d, v) in enumerate(pairs_list)],
    ["idx", "driver_id", "vehicle_id"]
)

# 3. Base trip level derived fields
base = (
    sampled_df
    .withColumn("trip_id", F.concat(F.lit("TRIP"), F.lpad(
        F.monotonically_increasing_id().cast("string"), 10, "0")))
    .withColumn("distance_km", F.round(F.col("trip_distance") * 1.60934, 2))
    .withColumn("pickup_zone_id", F.col("PULocationID"))
    .withColumn("dropoff_zone_id", F.col("DOLocationID"))
    .withColumn("status_rand", F.rand(seed=7))
    .withColumn("trip_status", F.when(F.col("status_rand") < 0.08, "CANCELLED").otherwise("COMPLETED"))
    .withColumn("surge_rand", F.rand(seed=11))
    .withColumn("surge_multiplier", F.when(F.col("surge_rand") < 0.8,
        F.lit(1.0)).otherwise(F.round(F.lit(1.1) + F.rand(seed=13) * 1.4, 2)))
    .withColumn("pair_idx", (F.rand(seed=201) * n_pairs).cast("int"))
    .join(F.broadcast(pairs_df), F.col("pair_idx") == F.col("idx"), "left")
    .drop("idx", "pair_idx")
    .withColumn("request_offset_sec", F.lit(120) + (F.rand(seed=21) * 360).cast("int"))
    .withColumn("event_request_ts", F.to_timestamp(F.from_unixtime(
        F.unix_timestamp("tpep_pickup_datetime") - F.col("request_offset_sec"))))
    .withColumn("assigned_offset_sec", F.lit(30) + (F.rand(seed=23) * 150)
        .cast("int"))
    .withColumn("event_assigned_ts", F.to_timestamp(F.from_unixtime(
        F.unix_timestamp("event_request_ts") + F.col("assigned_offset_sec"))))
    .withColumn("payment_offset_sec", F.lit(5) + (F.rand(seed=27) * 55)
        .cast("int"))
    .withColumn("event_payment_ts", F.to_timestamp(F.from_unixtime(
        F.unix_timestamp("tpep_dropoff_datetime") + F.col("payment_offset_sec"))))
    .withColumn("cancelled_offset_sec", F.lit(30) + (F.rand(seed=29) * 270)
        .cast("int"))
    .withColumn("event_cancelled_ts", F.to_timestamp(F.from_unixtime(
        F.unix_timestamp("event_assigned_ts") + F.col("cancelled_offset_sec"))))
)

# 4. Helper to build one event type
def make_event(df, event_type, ts_col, include_driver=True, include_fare=False):
    return df.select(
        F.expr("uuid()").alias("event_id"),
        F.lit(event_type).alias("event_type"),
        F.col(ts_col).alias("event_timestamp"),
        F.col("trip_id"),
        (F.col("driver_id") if include_driver else F.lit(None).cast("string")).alias("driver_id"),
        (F.col("vehicle_id") if include_driver else F.lit(None).cast("string")).alias("vehicle_id"),
        F.col("pickup_zone_id"),
        F.col("dropoff_zone_id"),
        F.col("tpep_pickup_datetime").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("distance_km"),
        (F.col("fare_amount") if include_fare else F.lit(None).cast("double")).alias("fare_amount"),
        (F.col("tip_amount") if include_fare else F.lit(None).cast("double")).alias("tip_amount"),
        (F.col("tolls_amount") if include_fare else F.lit(None).cast("double")).alias("toll_amount"),
        (F.col("total_amount") if include_fare else F.lit(None).cast("double")).alias("total_amount"),
        F.col("surge_multiplier"),
        F.col("trip_status"),
        (F.col("payment_type") if include_fare else F.lit(None).cast("long")).alias("payment_type"),
    )

# 5. Build lifecycle events
completed_base = base.filter(F.col("trip_status") == "COMPLETED")
cancelled_base = base.filter(F.col("trip_status") == "CANCELLED")

requested_events = make_event(base, "TRIP_REQUESTED", "event_request_ts", include_driver=False)
assigned_events = make_event(base, "DRIVER_ASSIGNED", "event_assigned_ts")
started_events = make_event(completed_base, "TRIP_STARTED", "tpep_pickup_datetime")
completed_events = make_event(completed_base, "TRIP_COMPLETED", "tpep_dropoff_datetime", include_fare=True)
payment_events = make_event(completed_base, "PAYMENT_COMPLETED", "event_payment_ts", include_fare=True)
cancelled_events = make_event(cancelled_base, "TRIP_CANCELLED", "event_cancelled_ts")

all_events = (
    requested_events
    .unionByName(assigned_events)
    .unionByName(started_events)
    .unionByName(completed_events)
    .unionByName(payment_events)
    .unionByName(cancelled_events)
)

# 6. Save as staging table
all_events.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.bronze.trip_events_staging")

result = spark.table("urban_mobility.bronze.trip_events_staging")
print("Total events generated:", result.count())
result.groupBy("event_type").count().orderBy("count", ascending=False).show()

In [0]:
%run "./anomaly_injector"

In [0]:
staged = spark.table("urban_mobility.bronze.trip_events_staging")

dirty_events = apply_all_bad_data(staged)
print("Total events after bad-data injection:", dirty_events.count())

sorted_events = dirty_events.orderBy("event_timestamp")

TMP_PATH = "/Volumes/urban_mobility/bronze/landing/_raw_batches/"
LANDING_PATH = "/Volumes/urban_mobility/bronze/landing/"

(sorted_events
 .write.mode("overwrite")
 .option("maxRecordsPerFile", 1000)
 .json(TMP_PATH))

print("Raw batch files written to temp folder.")

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

files = [f for f in dbutils.fs.ls(TMP_PATH) if f.name.endswith(".json")]
files_sorted = sorted(files, key=lambda f: f.name)

print("Total part files to rename:", len(files_sorted))

def rename_file(args):
    i, f = args
    batch_name = f"batch_{i:06d}.json"
    dbutils.fs.mv(f.path, LANDING_PATH + batch_name)
    return i

tasks = list(enumerate(files_sorted, start=1))

completed = 0
with ThreadPoolExecutor(max_workers=128) as executor:
    futures = [executor.submit(rename_file, t) for t in tasks]
    for future in as_completed(futures):
        future.result()
        completed += 1
        if completed % 1000 == 0:
            print(f"Renamed {completed}/{len(tasks)}")

dbutils.fs.rm(TMP_PATH, recurse=True)

landing_files = dbutils.fs.ls(LANDING_PATH)
print("Total batch files in landing:", len([f for f in landing_files if f.name.startswith("batch_")]))